# Download Album Covers (v1)
This notebook reads `spotify_411k.parquet`, downloads album cover images with a selectable size, and saves them under `data/covers/` using each song `track_id` as filename.

It implements **restart functionality**: by scanning the existing files, it only downloads what is missing.

In [1]:
from pathlib import Path
import os
from urllib.parse import urlparse
from urllib.request import Request, urlopen
import polars as pl
from tqdm.auto import tqdm

# --- CONFIGURATION ---
INPUT_PARQUET = Path("spotify_411k.parquet")
BASE_DIR = INPUT_PARQUET.parent
DATA_DIR = BASE_DIR / "data"
COVERS_DIR = DATA_DIR / "covers"

# COVER_SIZE: 0 for 640x640 (Large), 1 for 300x300 (Medium), 2 for 64x64 (Small)
COVER_SIZE_INDEX = 0 

COVERS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input Parquet: {INPUT_PARQUET.resolve()}")
print(f"Covers folder: {COVERS_DIR.resolve()}")

Input Parquet: /home/build/martin/spotify_v2/spotify_411k.parquet
Covers folder: /home/build/martin/spotify_v2/data/covers


In [2]:
# Load dataset
df_full = pl.read_parquet(INPUT_PARQUET).select(["track_id", "album"]).unique(subset=["track_id"])

# Restart logic: scan for already downloaded files
# Covers can have different extensions (.jpg, .png, etc.)
existing_covers = {p.stem for p in COVERS_DIR.glob("*") if p.is_file()}
print(f"Found {len(existing_covers):,} existing covers in {COVERS_DIR}.")

# Filter dataset to only include missing covers
df = df_full.filter(~pl.col("track_id").is_in(list(existing_covers)))

print(f"Total unique tracks in dataset: {df_full.height:,}")
print(f"Remaining covers to download: {df.height:,}")

# Create list of rows for remaining tracks
rows = df.to_dicts()

Found 35 existing covers in data/covers.
Total unique tracks in dataset: 411,000
Remaining covers to download: 410,965


In [3]:
def get_cover_url(album, size_index):
    if album is None or "images" not in album or not album["images"]:
        return None
    
    images = album["images"]
    if isinstance(images, list):
        # Spotify typically provides 3 sizes. Fallback if index is out of range.
        try:
            return images[size_index].get("url")
        except IndexError:
            return images[0].get("url")
    return None

def infer_image_extension(url: str) -> str:
    suffix = Path(urlparse(url).path).suffix.lower()
    if suffix in {".jpg", ".jpeg", ".png", ".webp"}:
        return suffix
    return ".jpg"

def download_binary(url: str, output_path: Path, timeout: int = 30) -> bool:
    if not url or not url.startswith("http"):
        return False
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urlopen(req, timeout=timeout) as resp:
            output_path.write_bytes(resp.read())
        return True
    except Exception:
        return False

In [4]:
# Download album cover images
stats = {"downloaded": 0, "failed": 0, "missing_url": 0, "already_exists": 0}

for row in tqdm(rows, desc="Downloading Covers"):
    track_id = row["track_id"]
    cover_url = get_cover_url(row["album"], COVER_SIZE_INDEX)

    if not cover_url:
        stats["missing_url"] += 1
        continue

    ext = infer_image_extension(cover_url)
    output_path = COVERS_DIR / (str(track_id) + ext)

    # Double check existence (case where file was added while script is running)
    if output_path.exists():
        stats["already_exists"] += 1
        continue

    if download_binary(cover_url, output_path):
        stats["downloaded"] += 1
    else:
        stats["failed"] += 1

print(stats)

{'downloaded': 410890, 'failed': 0, 'missing_url': 75, 'already_exists': 0}
